In [1]:
# ============================================================
# SETUP PATHS & IMPORTS
# ============================================================

import os
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Local path routing
# ------------------------------------------------------------
current_dir = os.getcwd()
workspace_root = current_dir

if os.path.basename(current_dir) == "notebooks":
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, "src")

sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# ------------------------------------------------------------
# Kaggle path routing
# ------------------------------------------------------------
kaggle_input = Path("/kaggle/input")

if kaggle_input.exists():
    for py_file in kaggle_input.rglob("*.py"):
        sys.path.insert(0, str(py_file.parent))

# ------------------------------------------------------------
# sklearn imports
# ------------------------------------------------------------
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------
# Optional Bayesian search imports
# ------------------------------------------------------------
try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical
    SKOPT_AVAILABLE = True
except Exception:
    BayesSearchCV = None
    Categorical = None
    SKOPT_AVAILABLE = False

# ------------------------------------------------------------
# Project imports
# ------------------------------------------------------------
try:
    from src.ev_data_utils import load_ev_data
    from src.ev_baseline_utils import (
        EVFeatureExtractor,
        LinearBaselineClassifier,
        competition_score,
        make_competition_scorer,
    )
    print("✅ Imports loaded from src/")
except ImportError:
    from ev_data_utils import load_ev_data
    from ev_baseline_utils import (
        EVFeatureExtractor,
        LinearBaselineClassifier,
        competition_score,
        make_competition_scorer,
    )
    print("✅ Imports loaded flattened")

print(f"SKOPT_AVAILABLE: {SKOPT_AVAILABLE}")

✅ Imports loaded from src/
SKOPT_AVAILABLE: True


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

TARGET_COL = "Will_Buy_EV"

RANDOM_STATE = 42
N_SPLITS = 2
HOLDOUT_SIZE = 0.3

MAKE_SUBMISSION = True

# ------------------------------------------------------------
# Search configuration
# ------------------------------------------------------------
SEARCH_MODE = "grid"        # "grid" or "bayesian"
N_ITER = 20                 # used only for Bayesian search
VERBOSE = 3
ERROR_SCORE = "raise"       # use "raise" for debugging

USE_BAYESIAN = SEARCH_MODE == "bayesian" and SKOPT_AVAILABLE

if SEARCH_MODE == "bayesian" and not SKOPT_AVAILABLE:
    print("⚠️ search_mode='bayesian' but scikit-optimize is unavailable.")
    print("⚠️ Falling back to grid search.")

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print(f"Search mode: {SEARCH_MODE}")
print(f"Using Bayesian search: {USE_BAYESIAN}")

Search mode: grid
Using Bayesian search: False


In [3]:
# ============================================================
# DATA LOADING
# ============================================================

train_df, test_df, data_source = load_ev_data(
    local_dir="data",
    train_name="train.csv",
    test_name="test.csv",
    sample_name="sample_ev.csv",
    target_col=TARGET_COL,
)

print(f"✅ Data source: {data_source}")
print(f"Train shape: {train_df.shape}")

if test_df is not None:
    print(f"Test shape: {test_df.shape}")

# ------------------------------------------------------------
# Target check
# ------------------------------------------------------------
if TARGET_COL not in train_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in train data.")

train_df = train_df.dropna(subset=[TARGET_COL]).copy()
train_df[TARGET_COL] = train_df[TARGET_COL].astype(int)

print("\nTarget distribution:")
print(train_df[TARGET_COL].value_counts())

assert set(train_df[TARGET_COL].unique()).issubset({0, 1}), (
    "Target must be binary 0/1. "
    "Coercion should happen inside ev_data_utils.load_ev_data."
)

# ------------------------------------------------------------
# Features / target
# ------------------------------------------------------------
X = train_df.drop(columns=[TARGET_COL], errors="ignore").copy()
y = train_df[TARGET_COL].astype(int).copy()

# ------------------------------------------------------------
# Optional holdout split
# ------------------------------------------------------------
if len(X) >= 50 and y.nunique() > 1:
    X_train, X_holdout, y_train, y_holdout = train_test_split(
        X,
        y,
        test_size=HOLDOUT_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    DO_HOLDOUT = True
else:
    X_train = X.copy()
    y_train = y.copy()
    X_holdout = None
    y_holdout = None
    DO_HOLDOUT = False

print(f"\nTrain rows used for fitting/search: {len(X_train)}")

if DO_HOLDOUT:
    print(f"Holdout rows: {len(X_holdout)}")
else:
    print("No holdout split.")

✅ Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\evpurchase_kaggle\data
✅ Data source: Train file: C:\Users\maran\OneDrive\Documents\Git Profile\Data-Projects\evpurchase_kaggle\data\train.csv
Train shape: (668665, 15)
Test shape: (286571, 14)

Target distribution:
Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64

Train rows used for fitting/search: 468065
Holdout rows: 200600


In [4]:
# ============================================================
# TWO SEPARATE MODELS
# ============================================================

UNIVARIATE_FEATURE = "Environmental_Concern_Level"

# ------------------------------------------------------------
# Model 1: Univariate linear baseline
# ------------------------------------------------------------
univariate_pipeline = Pipeline(
    [
        (
            "extractor",
            EVFeatureExtractor(
                feature_set="univariate",
                univariate_feature=UNIVARIATE_FEATURE,
                target_col=TARGET_COL,
                drop_id=True,
                impute_strategy="median",
                scale_numeric=True,
                onehot_categorical=True,
                add_derived_features=True,
            ),
        ),
        (
            "model",
            LinearBaselineClassifier(
                model_type="linear_regression",
                fit_intercept=True,
                clip_predictions=True,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# ------------------------------------------------------------
# Model 2: Multivariate linear baseline
# ------------------------------------------------------------
multivariate_pipeline = Pipeline(
    [
        (
            "extractor",
            EVFeatureExtractor(
                feature_set="multivariate",
                target_col=TARGET_COL,
                drop_id=True,
                impute_strategy="median",
                scale_numeric=True,
                onehot_categorical=True,
                add_derived_features=True,
            ),
        ),
        (
            "model",
            LinearBaselineClassifier(
                model_type="linear_regression",
                fit_intercept=True,
                clip_predictions=True,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

print("✅ Created two separate pipelines:")
print("1. Univariate linear baseline")
print("2. Multivariate linear baseline")

✅ Created two separate pipelines:
1. Univariate linear baseline
2. Multivariate linear baseline


In [5]:
# ============================================================
# COMPETITION SCORER
# ============================================================

scorer = make_competition_scorer(target_col=TARGET_COL)

print("✅ Using ROC AUC competition scorer.")

✅ Using ROC AUC competition scorer.


In [6]:
# ============================================================
# GRID SEARCH SPACES
# ============================================================

UNIVARIATE_GRID_SPACE = {
    # --------------------------------------------------------
    # Feature extraction parameters
    # --------------------------------------------------------
    "extractor__feature_set": ["univariate"],
    "extractor__univariate_feature": [
        "Environmental_Concern_Level",
        "Annual_Income_USD",
        "Daily_Commute_km",
        "Charging_Stations_Near_Home",
        "Charging_Stations_Near_Work",
        "Total_Charging_Stations",
        "High_Environmental_Concern",
    ],
    "extractor__impute_strategy": ["median", "mean"],
    "extractor__scale_numeric": [True, False],
    "extractor__onehot_categorical": [True],
    "extractor__add_derived_features": [True],

    # --------------------------------------------------------
    # Model parameters
    # --------------------------------------------------------
    "model__model_type": ["linear_regression"],
    "model__fit_intercept": [True],
    "model__clip_predictions": [True],
}

MULTIVARIATE_GRID_SPACE = {
    # --------------------------------------------------------
    # Feature extraction parameters
    # --------------------------------------------------------
    "extractor__feature_set": ["multivariate"],
    "extractor__add_derived_features": [True, False],
    "extractor__impute_strategy": ["median", "mean"],
    "extractor__scale_numeric": [True, False],
    "extractor__onehot_categorical": [True],

    # --------------------------------------------------------
    # Model parameters
    # --------------------------------------------------------
    "model__model_type": ["linear_regression"],
    "model__fit_intercept": [True],
    "model__clip_predictions": [True],
}

# ============================================================
# BAYESIAN SEARCH SPACES
# ============================================================

if SKOPT_AVAILABLE:

    UNIVARIATE_BAYESIAN_SPACE = {
        # ----------------------------------------------------
        # Feature extraction parameters
        # ----------------------------------------------------
        "extractor__feature_set": Categorical(["univariate"]),
        "extractor__univariate_feature": Categorical(
            [
                "Environmental_Concern_Level",
                "Annual_Income_USD",
                "Daily_Commute_km",
                "Charging_Stations_Near_Home",
                "Charging_Stations_Near_Work",
                "Total_Charging_Stations",
                "High_Environmental_Concern",
            ]
        ),
        "extractor__impute_strategy": Categorical(["median", "mean"]),
        "extractor__scale_numeric": Categorical([True, False]),
        "extractor__onehot_categorical": Categorical([True]),
        "extractor__add_derived_features": Categorical([True]),

        # ----------------------------------------------------
        # Model parameters
        # ----------------------------------------------------
        "model__model_type": Categorical(["linear_regression"]),
        "model__fit_intercept": Categorical([True]),
        "model__clip_predictions": Categorical([True]),
    }

    MULTIVARIATE_BAYESIAN_SPACE = {
        # ----------------------------------------------------
        # Feature extraction parameters
        # ----------------------------------------------------
        "extractor__feature_set": Categorical(["multivariate"]),
        "extractor__add_derived_features": Categorical([True, False]),
        "extractor__impute_strategy": Categorical(["median", "mean"]),
        "extractor__scale_numeric": Categorical([True, False]),
        "extractor__onehot_categorical": Categorical([True]),

        # ----------------------------------------------------
        # Model parameters
        # ----------------------------------------------------
        "model__model_type": Categorical(["linear_regression"]),
        "model__fit_intercept": Categorical([True]),
        "model__clip_predictions": Categorical([True]),
    }

else:

    UNIVARIATE_BAYESIAN_SPACE = None
    MULTIVARIATE_BAYESIAN_SPACE = None

# ============================================================
# SELECT ACTIVE SEARCH SPACES
# ============================================================

if USE_BAYESIAN:
    UNIVARIATE_SEARCH_SPACE = UNIVARIATE_BAYESIAN_SPACE
    MULTIVARIATE_SEARCH_SPACE = MULTIVARIATE_BAYESIAN_SPACE
    SEARCH_SPACE_KIND = "Bayesian"
else:
    UNIVARIATE_SEARCH_SPACE = UNIVARIATE_GRID_SPACE
    MULTIVARIATE_SEARCH_SPACE = MULTIVARIATE_GRID_SPACE
    SEARCH_SPACE_KIND = "Grid"

print(f"✅ Active search-space type: {SEARCH_SPACE_KIND}")

✅ Active search-space type: Grid


In [7]:
# ============================================================
# CV OBJECT
# ============================================================

CAN_CV = (
    y_train.nunique() > 1
    and len(y_train) >= 4
    and int(y_train.value_counts().min()) >= 2
)

if CAN_CV:
    min_class_count = int(y_train.value_counts().min())
    n_splits = max(2, min(N_SPLITS, min_class_count))

    cv_object = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    print(f"✅ Using StratifiedKFold with {n_splits} folds.")

else:
    cv_object = None
    print("⚠️ Not enough class diversity for stratified CV.")
    print("⚠️ Falling back to direct fit without search.")

✅ Using StratifiedKFold with 2 folds.


In [8]:
# ============================================================
# SEARCH EXECUTION
# ============================================================

if CAN_CV:

    if USE_BAYESIAN:

        univariate_search = BayesSearchCV(
            estimator=univariate_pipeline,
            search_spaces=UNIVARIATE_SEARCH_SPACE,
            n_iter=N_ITER,
            scoring=scorer,
            cv=cv_object,
            n_jobs=1,
            random_state=RANDOM_STATE,
            verbose=VERBOSE,
            refit=True,
            return_train_score=True,
            error_score=ERROR_SCORE,
        )

        multivariate_search = BayesSearchCV(
            estimator=multivariate_pipeline,
            search_spaces=MULTIVARIATE_SEARCH_SPACE,
            n_iter=N_ITER,
            scoring=scorer,
            cv=cv_object,
            n_jobs=1,
            random_state=RANDOM_STATE,
            verbose=VERBOSE,
            refit=True,
            return_train_score=True,
            error_score=ERROR_SCORE,
        )

    else:

        univariate_search = GridSearchCV(
            estimator=univariate_pipeline,
            param_grid=UNIVARIATE_SEARCH_SPACE,
            scoring=scorer,
            cv=cv_object,
            n_jobs=1,
            verbose=VERBOSE,
            refit=True,
            return_train_score=True,
            error_score=ERROR_SCORE,
        )

        multivariate_search = GridSearchCV(
            estimator=multivariate_pipeline,
            param_grid=MULTIVARIATE_SEARCH_SPACE,
            scoring=scorer,
            cv=cv_object,
            n_jobs=1,
            verbose=VERBOSE,
            refit=True,
            return_train_score=True,
            error_score=ERROR_SCORE,
        )

    print(f"🚀 Running {SEARCH_SPACE_KIND} search for univariate model...")
    univariate_search.fit(X_train, y_train)

    print(f"\n🚀 Running {SEARCH_SPACE_KIND} search for multivariate model...")
    multivariate_search.fit(X_train, y_train)

    UNIVARIATE_BEST_PIPELINE = univariate_search.best_estimator_
    MULTIVARIATE_BEST_PIPELINE = multivariate_search.best_estimator_

    UNIVARIATE_SEARCH_SCORE = float(univariate_search.best_score_)
    MULTIVARIATE_SEARCH_SCORE = float(multivariate_search.best_score_)

    print("\nUnivariate best params:")
    for k, v in univariate_search.best_params_.items():
        print(f"  {k}: {v}")

    print("\nMultivariate best params:")
    for k, v in multivariate_search.best_params_.items():
        print(f"  {k}: {v}")

else:

    print("⚠️ Skipping search and fitting pipelines directly.")

    univariate_pipeline.fit(X_train, y_train)
    multivariate_pipeline.fit(X_train, y_train)

    univariate_search = None
    multivariate_search = None

    UNIVARIATE_BEST_PIPELINE = univariate_pipeline
    MULTIVARIATE_BEST_PIPELINE = multivariate_pipeline

    UNIVARIATE_SEARCH_SCORE = competition_score(
        y_train,
        UNIVARIATE_BEST_PIPELINE.predict_proba(X_train)[:, 1],
    )

    MULTIVARIATE_SEARCH_SCORE = competition_score(
        y_train,
        MULTIVARIATE_BEST_PIPELINE.predict_proba(X_train)[:, 1],
    )

search_results = pd.DataFrame(
    [
        {
            "model": "univariate",
            "search_type": SEARCH_SPACE_KIND,
            "search_score_roc_auc": UNIVARIATE_SEARCH_SCORE,
        },
        {
            "model": "multivariate",
            "search_type": SEARCH_SPACE_KIND,
            "search_score_roc_auc": MULTIVARIATE_SEARCH_SCORE,
        },
    ]
)

print("\nSearch Results:")
print(search_results)

🚀 Running Grid search for univariate model...
Fitting 2 folds for each of 28 candidates, totalling 56 fits
[CV 1/2] END extractor__add_derived_features=True, extractor__feature_set=univariate, extractor__impute_strategy=median, extractor__onehot_categorical=True, extractor__scale_numeric=True, extractor__univariate_feature=Environmental_Concern_Level, model__clip_predictions=True, model__fit_intercept=True, model__model_type=linear_regression;, score=(train=0.843, test=0.844) total time=   4.9s
[CV 2/2] END extractor__add_derived_features=True, extractor__feature_set=univariate, extractor__impute_strategy=median, extractor__onehot_categorical=True, extractor__scale_numeric=True, extractor__univariate_feature=Environmental_Concern_Level, model__clip_predictions=True, model__fit_intercept=True, model__model_type=linear_regression;, score=(train=0.844, test=0.843) total time=   4.8s
[CV 1/2] END extractor__add_derived_features=True, extractor__feature_set=univariate, extractor__impute_str

KeyboardInterrupt: 

In [ ]:
# ============================================================
# HOLDOUT EVALUATION
# ============================================================

if DO_HOLDOUT:

    univariate_holdout_score = competition_score(
        y_holdout,
        UNIVARIATE_BEST_PIPELINE.predict_proba(X_holdout)[:, 1],
    )

    multivariate_holdout_score = competition_score(
        y_holdout,
        MULTIVARIATE_BEST_PIPELINE.predict_proba(X_holdout)[:, 1],
    )

    print("\nHoldout ROC AUC:")
    print(f"Univariate:   {univariate_holdout_score:.4f}")
    print(f"Multivariate: {multivariate_holdout_score:.4f}")

    UNIVARIATE_FINAL_SCORE = univariate_holdout_score
    MULTIVARIATE_FINAL_SCORE = multivariate_holdout_score

else:

    UNIVARIATE_FINAL_SCORE = UNIVARIATE_SEARCH_SCORE
    MULTIVARIATE_FINAL_SCORE = MULTIVARIATE_SEARCH_SCORE

if UNIVARIATE_FINAL_SCORE >= MULTIVARIATE_FINAL_SCORE:
    BEST_MODEL_NAME = "univariate"
    BEST_PIPELINE = UNIVARIATE_BEST_PIPELINE
else:
    BEST_MODEL_NAME = "multivariate"
    BEST_PIPELINE = MULTIVARIATE_BEST_PIPELINE

print(f"\n🏆 Best model by final ROC AUC: {BEST_MODEL_NAME}")
print(f"Univariate final score:   {UNIVARIATE_FINAL_SCORE:.4f}")
print(f"Multivariate final score: {MULTIVARIATE_FINAL_SCORE:.4f}")

In [ ]:
# ============================================================
# SAVE CV RESULTS WITH PARAMETERS
# ============================================================

cv_results_list = []

# Check if the search objects from the Grid/Bayesian search cell exist
if 'univariate_search' in locals() and univariate_search is not None:
    uni_cv_df = pd.DataFrame(univariate_search.cv_results_)
    uni_cv_df['model_architecture'] = 'univariate'
    cv_results_list.append(uni_cv_df)

if 'multivariate_search' in locals() and multivariate_search is not None:
    multi_cv_df = pd.DataFrame(multivariate_search.cv_results_)
    multi_cv_df['model_architecture'] = 'multivariate'
    cv_results_list.append(multi_cv_df)

if cv_results_list:
    full_cv_df = pd.concat(cv_results_list, ignore_index=True)
    
    # Move the architecture and score columns to the front for readability
    cols_to_front = ['model_architecture', 'mean_test_score', 'std_test_score', 'params']
    other_cols = [c for c in full_cv_df.columns if c not in cols_to_front]
    full_cv_df = full_cv_df[cols_to_front + other_cols]
    
    cv_path = results_dir / f"cv_results_with_params_{timestamp}.csv"
    full_cv_df.to_csv(cv_path, index=False)
    print(f"✅ CV results with parameters saved to: {cv_path}")
else:
    print("⚠️ No search objects found. If you used simple cross_val_score instead of Grid/BayesSearch, there are no hyperparameter grids to save.")

In [ ]:
# ============================================================
# HOLDOUT ERROR ANALYSIS BY PROMINENT CATEGORIES
# ============================================================

if DO_HOLDOUT and X_holdout is not None:
    print("🚀 Generating holdout error analysis...")
    
    # 1. Get predictions from the best pipeline
    holdout_proba = BEST_PIPELINE.predict_proba(X_holdout)[:, 1]
    holdout_preds = (holdout_proba >= 0.5).astype(int)
    
    # 2. Create a master evaluation dataframe
    eval_df = X_holdout.copy()
    eval_df['true_label'] = y_holdout.values
    eval_df['pred_label'] = holdout_preds
    eval_df['pred_proba'] = holdout_proba
    eval_df['is_correct'] = eval_df['true_label'] == eval_df['pred_label']
    
    # 3. Define the prominent categorical/ordinal features to analyze
    prominent_categories = [
        'Gender', 
        'City_Type', 
        'Current_Car_Type', 
        'Home_Charging_Possible', 
        'Subsidy_Available', 
        'Range_Anxiety_Level'
    ]
    
    # Filter to only categories that actually exist in the dataset
    valid_categories = [c for c in prominent_categories if c in eval_df.columns]
    
    breakdown_list = []
    
    # 4. Group by each category and calculate correct/wrong metrics
    for cat in valid_categories:
        grp = eval_df.groupby(cat).agg(
            total_samples=('is_correct', 'count'),
            correct_predictions=('is_correct', 'sum'),
            wrong_predictions=('is_correct', lambda x: (x == False).sum()),
            accuracy=('is_correct', 'mean'),
            actual_yes_count=('true_label', 'sum'),
            predicted_yes_count=('pred_label', 'sum')
        ).reset_index()
        
        grp['feature_name'] = cat
        grp.rename(columns={cat: 'feature_value'}, inplace=True)
        breakdown_list.append(grp)
        
    if breakdown_list:
        breakdown_df = pd.concat(breakdown_list, ignore_index=True)
        
        # Reorder columns for easy reading
        cols = [
            'feature_name', 'feature_value', 'total_samples', 
            'correct_predictions', 'wrong_predictions', 'accuracy', 
            'actual_yes_count', 'predicted_yes_count'
        ]
        breakdown_df = breakdown_df[cols]
        
        # Save the breakdown to CSV
        breakdown_path = results_dir / f"holdout_category_breakdown_{timestamp}.csv"
        breakdown_df.to_csv(breakdown_path, index=False)
        print(f"✅ Holdout category breakdown saved to: {breakdown_path}")
        
        # Display in notebook
        display(breakdown_df)
        
    else:
        print("⚠️ No valid categorical features found in X_holdout for breakdown.")
        
    # 5. Save the full row-level predictions for deep-dive debugging
    row_level_path = results_dir / f"holdout_row_level_predictions_{timestamp}.csv"
    eval_df.to_csv(row_level_path, index=False)
    print(f"\n✅ Full row-level holdout predictions (with true/pred labels) saved to: {row_level_path}")

else:
    print("⚠️ Holdout evaluation was not performed or X_holdout is unavailable.")